# Worksheet on the invalid curve attack

This worksheet will guide you through the invalid curve attack using real-world parameters.

We use the elliptic curve $$E: y^2 = x^3 - 3x + 2455155546008943817740293915197451784769108058161191238065$$
over a finite field with $\mathbb{F}_p$ where $p$ is a $192$-bit prime. The elliptic curve is known as `secp192r1`.
The SageMath code for this setup is provided below (taken from https://std.neuromancer.sk/nist/P-192). 

In [1]:
p = 0xfffffffffffffffffffffffffffffffeffffffffffffffff
K = GF(p)
a = K(0xfffffffffffffffffffffffffffffffefffffffffffffffc)
b = K(0x64210519e59c80e70fa7e9ab72243049feb8deecc146b9b1)
E = EllipticCurve(K, (a, b))
P = E(0x188da80eb03090f67cbf20eb43a18800f4ff0afd82ff1012, 0x07192b95ffc8da78631011ed6b24cdd573f977a11e794811)

You can check that the order of $E$ is prime, we denote $\#E = q$. The point $P$ defined above is a public parameter, and part of the setup of the Diffie-Hellman key exchange.

In the following, you will communicate with Alice, who uses a secret key $\alpha \in \{1, \dots, q-1\}$. Your communication with Alice is modeled by the following oracle which on input a point $P$ returns $[\alpha] P$. Note: the oracle does not check if the point is on the correct curve. It only uses the coefficient $a=-3$ of the elliptic curve for addition. 

*The oracle should be viewed as a black box for now*

In [3]:
def oracle(x,y,level=1):
    """
    INPUT:
        -  coordinates (x,y) of a point P = (x,y) on the elliptic curve 
            E: y^2 = x^3 - 3*x + b over F_p with
            p = 0xfffffffffffffffffffffffffffffffeffffffffffffffff
            and b arbitrary
        -  level: 1, 2 or 3  (optional, the default value is 1)
            The level determines the strength of the secret key.
            For level 1, the secret key is a 3 digit number (10 bits);
            for level 2, the secret key is a 7 digit number (24 bits);
            for level 3, the bitlength of the secret key is not provided
    
    OUTPUT:
        coordinates of a point Q = (new_x : new_y : new_z) so that 
        Q = c * P for some  fixed secret integer c (depending on the level)
    """
    t = 1
    k = 4^level
    F = x.parent()
    if level == 2:
        k = k - 6
    r, s = x, y
    u = 18*r^4*s-16*r*s^3-36*r^2*s+18*s
    v = -27*r^6+36*r^3*s^2+81*r^4-8*s^4-36*r*s^2-81*r^2+27
    w = 8*s^3
    for i in range(k):
        U = -46438023168*u^22*v*w+227030335488*u^19*v^3*w^2+510818254848*u^20*v*w^3-454060670976*u^16*v^5*w^3-2043273019392*u^17*v^3*w^4+440301256704*u^13*v^7*w^4-2554091274240*u^18*v*w^5+3137146454016*u^14*v^5*w^5-163074539520*u^10*v^9*w^5+8173092077568*u^15*v^3*w^6-1981355655168*u^11*v^7*w^6-43486543872*u^7*v^11*w^6+7662273822720*u^16*v*w^7-9246326390784*u^12*v^5*w^7+81537269760*u^8*v^9*w^7+48318382080*u^4*v^13*w^7-19070548180992*u^13*v^3*w^8+3302259425280*u^9*v^7*w^8+391378894848*u^5*v^11*w^8-8589934592*u*v^15*w^8-15324547645440*u^14*v*w^9+15025280385024*u^10*v^5*w^9+766450335744*u^6*v^9*w^9-96636764160*u^2*v^13*w^9+28605822271488*u^11*v^3*w^10-2201506283520*u^7*v^7*w^10-456608710656*u^3*v^11*w^10+21454366703616*u^12*v*w^11-14447384985600*u^8*v^5*w^11-1157829230592*u^4*v^9*w^11+4831838208*v^13*w^11-28605822271488*u^9*v^3*w^12+108716359680*u*v^11*w^12-21454366703616*u^10*v*w^13+8090535591936*u^6*v^5*w^13+505531072512*u^2*v^9*w^13+19070548180992*u^7*v^3*w^14+660451885056*u^3*v^7*w^14+15324547645440*u^8*v*w^15-2311581597696*u^4*v^5*w^15-32614907904*v^9*w^15-8173092077568*u^5*v^3*w^16-220150628352*u*v^7*w^16-7662273822720*u^6*v*w^17+165112971264*u^2*v^5*w^17+2043273019392*u^3*v^3*w^18+2554091274240*u^4*v*w^19+41278242816*v^5*w^19-227030335488*u*v^3*w^20-510818254848*u^2*v*w^21+46438023168*v*w^23
        V = +34828517376*u^24-185752092672*u^21*v^2*w-417942208512*u^22*w^2+412782428160*u^18*v^4*w^2+1857520926720*u^19*v^2*w^3-550376570880*u^15*v^6*w^3+2298682146816*u^20*w^4-3405455032320*u^16*v^4*w^4+550376570880*u^12*v^8*w^4-8358844170240*u^17*v^2*w^5+3852635996160*u^13*v^6*w^5-423993802752*u^9*v^10*w^5-7662273822720*u^18*w^6+12383472844800*u^14*v^4*w^6-3302259425280*u^10*v^8*w^6+202937204736*u^6*v^12*w^6+22290251120640*u^15*v^2*w^7-11557907988480*u^11*v^6*w^7+1891664658432*u^7*v^10*w^7-38654705664*u^3*v^14*w^7+17240116101120*u^16*w^8-26005292974080*u^12*v^4*w^8+7705271992320*u^8*v^8*w^8-456608710656*u^4*v^12*w^8-2147483648*v^16*w^8-39007939461120*u^13*v^2*w^9+19263179980800*u^9*v^6*w^9-2543962816512*u^5*v^10*w^9-19327352832*u*v^14*w^9-27584185761792*u^14*w^10+34673723965440*u^10*v^4*w^10-8806025134080*u^6*v^8*w^10+43486543872*u^2*v^12*w^10+46809527353344*u^11*v^2*w^11-19263179980800*u^7*v^6*w^11+1108906868736*u^3*v^10*w^11+32181550055424*u^12*w^12-30339508469760*u^8*v^4*w^12+4953389137920*u^4*v^8*w^12+14495514624*v^12*w^12-39007939461120*u^9*v^2*w^13+11557907988480*u^5*v^6*w^13-32614907904*u*v^10*w^13-27584185761792*u^10*w^14+17336861982720*u^6*v^4*w^14-1100753141760*u^2*v^8*w^14+22290251120640*u^7*v^2*w^15-3852635996160*u^3*v^6*w^15+17240116101120*u^8*w^16-6191736422400*u^4*v^4*w^16-8358844170240*u^5*v^2*w^17+550376570880*u*v^6*w^17-7662273822720*u^6*w^18+1238347284480*u^2*v^4*w^18+1857520926720*u^3*v^2*w^19+2298682146816*u^4*w^20-103195607040*v^4*w^20-185752092672*u*v^2*w^21-417942208512*u^2*w^22+34828517376*w^24
        W = -82556485632*u^18*v^3*w^3+330225942528*u^15*v^5*w^4+743008370688*u^16*v^3*w^5-513684799488*u^12*v^7*w^5-2311581597696*u^13*v^5*w^6+391378894848*u^9*v^9*w^6-2972033482752*u^14*v^3*w^7+2641807540224*u^10*v^7*w^7-152202903552*u^6*v^11*w^7+6934744793088*u^11*v^5*w^8-1369826131968*u^7*v^9*w^8+28991029248*u^3*v^13*w^8+6934744793088*u^12*v^3*w^9-5503765708800*u^8*v^7*w^9+326149079040*u^4*v^11*w^9-2147483648*v^15*w^9-11557907988480*u^9*v^5*w^10+1761205026816*u^5*v^9*w^10-28991029248*u*v^13*w^10-10402117189632*u^10*v^3*w^11+5870683422720*u^6*v^7*w^11-195689447424*u^2*v^11*w^11+11557907988480*u^7*v^5*w^12-978447237120*u^3*v^9*w^12+10402117189632*u^8*v^3*w^13-3302259425280*u^4*v^7*w^13+21743271936*v^11*w^13-6934744793088*u^5*v^5*w^14+195689447424*u*v^9*w^14-6934744793088*u^6*v^3*w^15+880602513408*u^2*v^7*w^15+2311581597696*u^3*v^5*w^16+2972033482752*u^4*v^3*w^17-73383542784*v^7*w^17-330225942528*u*v^5*w^18-743008370688*u^2*v^3*w^19+82556485632*v^3*w^21
        A = +1492992*r^13-1990656*r^10*s^2*t-8957952*r^11*t^2-2654208*r^7*s^4*t^2+3981312*r^8*s^2*t^3+4718592*r^4*s^6*t^3+22394880*r^9*t^4+15925248*r^5*s^4*t^4-1572864*r*s^8*t^4+3981312*r^6*s^2*t^5-9437184*r^2*s^6*t^5-29859840*r^7*t^6-23887872*r^3*s^4*t^6-15925248*r^4*s^2*t^7+1179648*s^6*t^7+22394880*r^5*t^8+10616832*r*s^4*t^8+13934592*r^2*s^2*t^9-8957952*r^3*t^10-3981312*s^2*t^11+1492992*r*t^12
        B = +4478976*r^12*s-9953280*r^9*s^3*t-26873856*r^10*s*t^2+7962624*r^6*s^5*t^2+39813120*r^7*s^3*t^3-3538944*r^3*s^7*t^3+67184640*r^8*s*t^4-21233664*r^4*s^5*t^4+1048576*s^9*t^4-59719680*r^5*s^3*t^5+7077888*r*s^7*t^5-89579520*r^6*s*t^6+18579456*r^2*s^5*t^6+39813120*r^3*s^3*t^7+67184640*r^4*s*t^8-5308416*s^5*t^8-9953280*r*s^3*t^9-26873856*r^2*s*t^10+4478976*s*t^12
        C = +1492992*r^12*t-5971968*r^9*s^2*t^2-8957952*r^10*t^3+7962624*r^6*s^4*t^3+23887872*r^7*s^2*t^4-3538944*r^3*s^6*t^4+22394880*r^8*t^5-15925248*r^4*s^4*t^5-35831808*r^5*s^2*t^6-29859840*r^6*t^7+7962624*r^2*s^4*t^7+23887872*r^3*s^2*t^8+22394880*r^4*t^9-5971968*r*s^2*t^10-8957952*r^2*t^11+1492992*t^13
        r,s,t,u,v,w = A,B,C,U,V,W
    new_x1,new_y1,new_z1,new_x2,new_y2,new_z2 = r,s,t,u,v,w
    new_x3 = (-W^3*A^3+1*U*W^2*A^2*C+1*W^3*B^2*C+1*U^2*W*A*C^2-2*V*W^2*B*C^2-1*U^3*C^3+1*V^2*W*C^3)*(-C*U + A*W)
    new_y3 = (W^4*A^3*B-2*V*W^3*A^3*C-1*W^4*B^3*C+3*U*V*W^2*A^2*C^2-3*U^2*W^2*A*B*C^2+3*V*W^3*B^2*C^2+2*U^3*W*B*C^3-3*V^2*W^2*B*C^3-1*U^3*V*C^4+1*V^3*W*C^4)
    new_z3 = (W^4*A^3*C-3*U*W^3*A^2*C^2+3*U^2*W^2*A*C^3-1*U^3*W*C^4)
    try:
        Q = ProjectiveSpace(F,2)(new_x3,new_y3,new_z3)
    except:
        try:
            if new_z1*new_z2 == 0:
                Q = ProjectiveSpace(F,2)(new_x2*new_z2 + new_x1*new_z1, new_y2*new_z2 + new_y1*new_z1, new_z2**2 + new_z1**2)
            else:
                Q = ProjectiveSpace(F,2)((9*new_x1^4 - 8*new_x1*new_y1^2*new_z1 - 18*new_x1^2*new_z1^2 + 9*new_z1^4)*2*new_y1*new_z1 , -27*new_x1^6 + 36*new_x1^3*new_y1^2*new_z1 + 81*new_x1^4*new_z1^2 - 8*new_y1^4*new_z1^2 - 36*new_x1*new_y1^2*new_z1^3 - 81*new_x1^2*new_z1^4 + 27*new_z1^6, 8*new_y1^3*new_z1^3)
        except:
            raise ValueError("Sorry, the oracle cannot handle this case. Please try a different input.")
    if not Q[2].is_zero():
        return ProjectiveSpace(2)(Q[0]/Q[2],Q[1]/Q[2],1)
    else:
        return ProjectiveSpace(2)(0,1,0)

## Exercise 1

In this exercise, we study elliptic curves $E_{b'} : y^2 = x^3 - 3x +b' $ for varying values of $b'$. 

**(a)** Choose a random element $b' \in \mathbb{F}_p$, and construct the elliptic curve $E_{b'}$. Compute the number of points of $E_b'$. Is $E_{b'}$ isomorphic to $E$?  


**(b)** Find values $b_{k} \in \mathbb{F}_p$ so that the order $\# E_{b_k}(\mathbb{F}_p)$ is divisble by $k$ for $k \in \{2, 3, 5, 7, 11\}$.


**(c)** For the values $b_{k}$ computed above, find points $P_{k} \in E_{b_k}(\mathbb{F}_p)$ of order $k$  for $k \in \{2,3,5,7,11\}$.

## Exercise 2

At first, Alice chooses a weak secret key $\alpha \in \{1, \dots, 1000\}$. And publishes the following public key $A  \in E(\mathbb{F}_p)$.

In [4]:
A = E([6127092148455568788859865970055734976601092954612945364416, 1332437165805441346373759823549808425907890810335908434092]); A


(6127092148455568788859865970055734976601092954612945364416 : 1332437165805441346373759823549808425907890810335908434092 : 1)

**(a)** Use the oracle on input $P$ to check that the public key is correct. More precisely, call `oracle(x_1,y_1,level=1)`, where $P = (x_1,y_1)$, equivalently call `oracle(*P.xy(),level=1)`.

**(b)** Find the value of  $\alpha \pmod{2}$. 

*Hint: Use the point $P_2$ constructed in the previous exercise, and query the oracle provided above using the key word `level=1`. More precisely, call `oracle(x_2,y_2,level=1)`, where $P_2 = (x_2,y_2)$.*

**(c)** In a similar way, find $\alpha \pmod{k}$ for $k \in \{2,3,5,7,11\}$.  

**(d)** Use the Chinese remainder theorem to compute $\alpha \pmod{2\cdot 3\cdot 5\cdot 7\cdot 11}$. Using the restriction  $\alpha \in \{1, \dots, 1000\}$, deduce the correct value of $\alpha$. 

**(e)** In this exercise, the value for $\alpha$ is so small that you can easily find it by brute force, i.e. simply testing all values $\alpha \in \{1, \dots, 1000\}$. Check that you obtain the same result.

## Exercise 3
Now Alice improves her choice of secret key. She chooses a secret key with $7$ digits, i.e. $\alpha \in \{1, \dots, 10^{7}\}$. Her public key is `A = E([1716173386887646721522212999132904578695759391808627156141,
 1974800061378766962161712262235833926389829240712486018466])`. 

**(a)** Determine a bound $B \in \mathbb{N}$ so that $\prod_{p < B \text{ prime}} p > 10^{7}$.

*Hint: You can use the function `prime_range()` in SageMath. For instance `prime_range(N)` for some integer $N$ provides a list of all primes smaller than $N$.*

**(b)** Find $\alpha \pmod{p_i}$ for each prime $p_i < B$, and use the Chinese Remainder Theorem to deduce $\alpha$.

*For this exercise, you need to call the oracle with keyword `level=2`*. You can check that `A == E(oracle(*P.xy(),level=2)` holds.

**(c)** Is it still possible to find the result by brute force? Time the computations using `%time`.

## Exercise 4
To avoid any brute force attacks, Alice chooses a secret key $\alpha \in \{1, \dots, q-1\}$ now. Use the same techniques as above to find her secret key. 

The public key $A = [\alpha] P$ is provided below, and the oracle has to be called with keyword `level=3`.

*Hint: You don't need to find the residue for every prime up to some bound $B$, but instead it might be faster to find the residues for a set of "small" primes $S$ so that $\prod_{p_k \in S} p_k > p$.*

In [10]:
A = E([5734506099364995637721188567771987633115050105648405605804, 2893165864442171698077386145699542901889637492549747973376])